# prep.tableS3

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from daforfer import DaforferDB
from skbio.diversity.alpha import chao1
import matplotlib.pyplot as plt
import networkx as nx
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

┌─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  name   │                                                        description                                                        │
│ varchar │                                                          varchar                                                          │
├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ TableS1 │ Table S1: Library sites and context                                                                                       │
│ TableS2 │ This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc. │
└─────────┴───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

## PAB diversity

In [8]:
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left')
bacteria_hits

,site,library,habitat,n_extracts,host_taxon,taxid,scientific_name,is_pab,pab_type
0,C1,PV534,Crop,3,Diplotaxis erucoides,NaN,NaN,NaN,NaN
1,C1,PV535,Crop,17,Brassica oleracea,1563157.0,Pseudomonas endophytica,True,pab_unknown
2,C1,PV535,Crop,17,Brassica oleracea,1270.0,Micrococcus luteus,True,pab_unknown
3,C1,PV538,Crop,8,Brassica oleracea,NaN,NaN,NaN,NaN
4,C1,PV540,Crop,1,Picris echioides,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
626,Z2,PV527,Crop,4,Convolvulus arvensis,1770058.0,Devosia elaeis,True,pab_unknown
627,Z2,PV529,Crop,1,Picris echioides,47880.0,Pseudomonas fulva,True,pab_unknown
628,Z2,PV529,Crop,1,Picris echioides,1220495.0,Pseudomonas punonensis,True,pab_unknown
629,Z2,PV529,Crop,1,Picris echioides,289370.0,Pseudomonas argentinensis,True,pab_unknown


We simply create a list of item counts in one of the columns. 

In [9]:
pab_alpha_diversity = bacteria_hits.value_counts(
    ['site', 'habitat', 'scientific_name']
    ).reset_index().groupby(
        ['site', 'habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
pab_alpha_diversity

,site,habitat,hits
0,C1,Crop,"[1, 1, 1, 1, 1, 1]"
1,C2,Crop,"[2, 1, 1, 1, 1, 1, 1, 1]"
2,E1,Wasteland,"[2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,E2,Wasteland,"[3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4,E3,Wasteland,"[5, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
5,E4,Wasteland,"[8, 4, 4, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, ..."
6,H1,Crop,[1]
7,H2,Crop,[1]
8,H3,Crop,"[1, 1, 1, 1]"
9,L1,Edge,"[3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [10]:
pab_alpha_diversity['species_richness'] = pab_alpha_diversity['hits'].apply(lambda x: len(x))
pab_alpha_diversity['chao1'] = pab_alpha_diversity['hits'].apply(chao1)
pab_alpha_diversity = pab_alpha_diversity.sort_values(by=['habitat', 'site'])
pab_alpha_diversity['disturbed'] = pab_alpha_diversity['habitat'].apply(
    lambda x: {"Crop": "disturbed", "Wasteland": "non-disturbed", "Edge": "disturbed", "Oak": "non-disturbed"}[x]
)
pab_alpha_diversity = pab_alpha_diversity.drop(columns=['hits'])[['site', 'habitat', 'disturbed', 'species_richness', 'chao1']]
pab_alpha_diversity

,site,habitat,disturbed,species_richness,chao1
0,C1,Crop,disturbed,6,21.000000
1,C2,Crop,disturbed,8,18.500000
6,H1,Crop,disturbed,1,1.000000
7,H2,Crop,disturbed,1,1.000000
8,H3,Crop,disturbed,4,10.000000
13,M1,Crop,disturbed,2,2.000000
14,M2,Crop,disturbed,1,1.000000
15,M3,Crop,disturbed,2,3.000000
16,M4,Crop,disturbed,1,1.000000
21,Z1,Crop,disturbed,11,33.500000


## Virus diversity

In [11]:
# virus_hits = pd.read_csv("output/hits.virus.csv", sep=";")
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left')
virus_alpha_diversity = virus_hits.value_counts(
    ['site', 'habitat', 'scientific_name']
    ).reset_index().groupby(
        ['site', 'habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
virus_alpha_diversity['species_richness'] = virus_alpha_diversity['hits'].apply(lambda x: len(x))
virus_alpha_diversity['chao1'] = virus_alpha_diversity['hits'].apply(chao1)
virus_alpha_diversity = virus_alpha_diversity.drop(columns=['hits'])[['site', 'habitat',  'species_richness', 'chao1']]
virus_alpha_diversity


,site,habitat,species_richness,chao1
0,C1,Crop,18,27.166667
1,C2,Crop,12,16.200000
2,E1,Wasteland,14,23.000000
3,E2,Wasteland,30,99.000000
4,E3,Wasteland,17,47.333333
5,E4,Wasteland,28,66.250000
6,H1,Crop,13,18.600000
7,H2,Crop,12,15.750000
8,H3,Crop,33,58.666667
9,L1,Edge,54,65.400000


## Plant diversity

In [12]:
plant_hits = pd.read_csv("input/hits.plant.csv", sep=';', header=None, names=['Code', 'Species name']).dropna()
plant_hits['Collection_code'] = plant_hits['Code'].apply(lambda x: x.split('.')[0])
plant_hits['position'] = plant_hits['Code'].apply(lambda x: int(x.split('.')[1]))
plant_hits = pd.merge(
    pd.read_csv("input/mcleish24.TableS1.csv", sep=';'), plant_hits, on='Collection_code',
    how='left'
)
plant_hits = plant_hits[['Site_code', 'Species name']].drop_duplicates(['Site_code', 'Species name']).rename(columns={'Site_code': 'site'})
plant_hits = pd.merge(
    metadata[['site', 'habitat']].drop_duplicates(['site', 'habitat'], keep='first'), 
    plant_hits, on='site', how='left'
).dropna(subset='Species name')
plant_alpha_diversity = plant_hits.value_counts(
    ['site', 'habitat', 'Species name']
    ).reset_index().groupby(
        ['site', 'habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
plant_alpha_diversity['species_richness'] = plant_alpha_diversity['hits'].apply(lambda x: len(x))
plant_alpha_diversity['chao1'] = plant_alpha_diversity['hits'].apply(chao1)
plant_alpha_diversity = plant_alpha_diversity.drop(columns=['hits'])[['site', 'habitat',  'species_richness', 'chao1']]
plant_alpha_diversity

,site,habitat,species_richness,chao1
0,C1,Crop,16,136.0
1,C2,Crop,22,253.0
2,E1,Wasteland,46,1081.0
3,E2,Wasteland,56,1596.0
4,E3,Wasteland,58,1711.0
5,E4,Wasteland,46,1081.0
6,H1,Crop,6,21.0
7,H2,Crop,5,15.0
8,H3,Crop,9,45.0
9,L1,Edge,36,666.0


## Host diversity

In [13]:
host_alpha_diversity = metadata.value_counts(
    ['site', 'habitat', 'host_taxon']
).reset_index().groupby(
    ['site', 'habitat']
)['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
host_alpha_diversity['species_richness'] = host_alpha_diversity['hits'].apply(lambda x: len(x))
host_alpha_diversity['chao1'] = host_alpha_diversity['hits'].apply(chao1)
host_alpha_diversity = host_alpha_diversity.drop(columns=['hits'])[['site', 'habitat',  'species_richness', 'chao1']]
host_alpha_diversity

,site,habitat,species_richness,chao1
0,C1,Crop,4,5.500000
1,C2,Crop,5,8.000000
2,E1,Wasteland,14,105.000000
3,E2,Wasteland,18,44.250000
4,E3,Wasteland,13,31.333333
5,E4,Wasteland,18,86.000000
6,H1,Crop,3,3.500000
7,H2,Crop,3,3.500000
8,H3,Crop,4,5.500000
9,L1,Edge,28,45.000000


## Merge

In [14]:
# bacteria_alpha_diversity = db.conn.query('SELECT * FROM D_PABAlphaDiv').df()
alpha_diversity = pd.merge(
    pab_alpha_diversity[['site', 'habitat', 'disturbed', 'species_richness', 'chao1']], 
    virus_alpha_diversity[['site', 'species_richness', 'chao1']],
    on='site', suffixes=['_bact', '_vir']
)

alpha_diversity = pd.merge(
    alpha_diversity,
    plant_alpha_diversity[['site', 'species_richness', 'chao1']].rename(
        columns={
            'species_richness': 'species_richness_plant',
            'chao1': 'chao1_plant',
        }
    ),
    on='site'
)

alpha_diversity = pd.merge(
    alpha_diversity,
    host_alpha_diversity[['site', 'species_richness', 'chao1']].rename(
        columns={
            'species_richness': 'species_richness_host',
            'chao1': 'chao1_host',
        }
    ),
    on='site'
)


# alpha_diversity.to_csv("output/diversity.all.csv", sep=";", index=False)
# db.save_dataframe(
#     df=alpha_diversity, table_name="D_ADAllOrganismsSite",
#     description="Alpha diversity per site for virus, plant, host and bacteria at site level"
# )

alpha_diversity

,site,habitat,disturbed,species_richness_bact,chao1_bact,species_richness_vir,chao1_vir,species_richness_plant,chao1_plant,species_richness_host,chao1_host
0,C1,Crop,disturbed,6,21.000000,18,27.166667,16,136.0,4,5.500000
1,C2,Crop,disturbed,8,18.500000,12,16.200000,22,253.0,5,8.000000
2,H1,Crop,disturbed,1,1.000000,13,18.600000,6,21.0,3,3.500000
3,H2,Crop,disturbed,1,1.000000,12,15.750000,5,15.0,3,3.500000
4,H3,Crop,disturbed,4,10.000000,33,58.666667,9,45.0,4,5.500000
5,M1,Crop,disturbed,2,2.000000,20,23.000000,14,105.0,5,11.000000
6,M2,Crop,disturbed,1,1.000000,13,31.333333,9,45.0,4,10.000000
7,M3,Crop,disturbed,2,3.000000,9,11.000000,11,66.0,4,5.500000
8,M4,Crop,disturbed,1,1.000000,9,23.000000,15,120.0,3,3.500000
9,Z1,Crop,disturbed,11,33.500000,14,21.000000,13,91.0,6,21.000000


## Cooccurrences by site

In [15]:
cooccurrence_network = nx.read_graphml("output/network.coocurrence.virusbact-bylibrary.trans.graphml")
cooccurrence_network.number_of_edges()

57

We need to use original hit tables, as we will use the OTUs scientific names to make subgraphs on the cooccurrence network.

In [16]:
hits_by_site = pd.concat([
    bacteria_hits[['site', 'habitat', 'scientific_name']],
    virus_hits[['site', 'habitat', 'scientific_name']],
]).drop_duplicates(['site', 'habitat', 'scientific_name'], keep='first')
hits_by_site

,site,habitat,scientific_name
0,C1,Crop,NaN
1,C1,Crop,Pseudomonas endophytica
2,C1,Crop,Micrococcus luteus
5,C1,Crop,Sphingomonas sp. Leaf20
6,C1,Crop,Rhodococcoides fascians
...,...,...,...
1658,Z2,Crop,Pepper mild mottle virus
1660,Z2,Crop,Barley yellow dwarf virus - GAV
1664,Z2,Crop,Tomato aspermy virus RNA 3
1665,Z2,Crop,Tobacco mild green mosaic virus




We look for subgraphs of the cooccurrence graph within each of the sites

In [17]:
cooccurrences_by_site = []
for site in hits_by_site.site.unique():
    site_cooccurrence_subnetwork = cooccurrence_network.subgraph(
        hits_by_site.query('site == "{:s}"'.format(site))['scientific_name'].to_list()
    )
    cooccurrences_by_site.append({
        "site": site,
        "total_cooccurrences": site_cooccurrence_subnetwork.number_of_edges()
    })
cooccurrences_by_site = pd.DataFrame.from_records(cooccurrences_by_site)
cooccurrences_by_site = pd.merge(
    cooccurrences_by_site, # type: ignore
    hits_by_site.drop_duplicates(['site', 'habitat'], keep='first')[['site', 'habitat']]
)
cooccurrences_by_site

,site,total_cooccurrences,habitat
0,C1,5,Crop
1,C2,6,Crop
2,E1,8,Wasteland
3,E2,5,Wasteland
4,E3,11,Wasteland
5,E4,21,Wasteland
6,H1,0,Crop
7,H2,2,Crop
8,H3,5,Crop
9,L1,34,Edge


In [18]:
tableS3 = pd.merge(alpha_diversity, cooccurrences_by_site, on=['site', 'habitat'])
tableS3

,site,habitat,disturbed,species_richness_bact,chao1_bact,species_richness_vir,chao1_vir,species_richness_plant,chao1_plant,species_richness_host,chao1_host,total_cooccurrences
0,C1,Crop,disturbed,6,21.000000,18,27.166667,16,136.0,4,5.500000,5
1,C2,Crop,disturbed,8,18.500000,12,16.200000,22,253.0,5,8.000000,6
2,H1,Crop,disturbed,1,1.000000,13,18.600000,6,21.0,3,3.500000,0
3,H2,Crop,disturbed,1,1.000000,12,15.750000,5,15.0,3,3.500000,2
4,H3,Crop,disturbed,4,10.000000,33,58.666667,9,45.0,4,5.500000,5
5,M1,Crop,disturbed,2,2.000000,20,23.000000,14,105.0,5,11.000000,3
6,M2,Crop,disturbed,1,1.000000,13,31.333333,9,45.0,4,10.000000,1
7,M3,Crop,disturbed,2,3.000000,9,11.000000,11,66.0,4,5.500000,4
8,M4,Crop,disturbed,1,1.000000,9,23.000000,15,120.0,3,3.500000,3
9,Z1,Crop,disturbed,11,33.500000,14,21.000000,13,91.0,6,21.000000,12


## Save

In [19]:
db.save_dataframe(
    tableS3, "D_Site_level_div",
    description="Site-level diversity and number of cooccurring virus-bacteria"
)

Saved D_Site_level_div to db.2025-10-27


In [20]:
si.save_dataframe(
    tableS3, "TableS3",
    description="Site-level diversity and number of cooccurring virus-bacteria"
)

Saved TableS3 to si.2025-10-27


In [21]:
tableS3.to_csv("output/TableS3.csv", sep=";")

In [22]:
si.conn.close()
db.conn.close()